In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

In [2]:
# Для генерации случайных разбиений и инициализации моделей
random_seeds = [1, 42, 100, 200, 500, 1000, 2000, 3000, 4000, 5000]

In [3]:
# грузим датасет, удаляем объявления вне мкада и те, к которым внутри МКАДа не подтянулся АО
dataset = pd.read_excel('../Dataset/objects_w_geo_prepared_V0_2j.xlsx')
dataset = dataset.loc[dataset['inMKAD']==1].reset_index(drop=True)
dataset = dataset.loc[~(dataset['AO']=='НЕОПР')].reset_index(drop=True)
dataset = dataset.loc[dataset['office_class'].isin(['C'])].reset_index(drop=True)
dataset

,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,TotalOfficeCnt_2000,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter
0,28,7.0,1357,9500,C,1,1,0,0,0,...,557,0,35,6,0,0,1,CВАО,0,7592.842639
1,31,12.3,976,12000,C,3,4,0,1,0,...,708,0,1,77,0,0,1,ЮВАО,0,5449.520429
2,99,12.0,1167,14000,C,3,3,0,0,0,...,261,0,8,46,0,0,1,СЗАО,0,7381.297810
3,109,15.0,1100,16500,C,2,4,1,0,1,...,703,18,3,313,0,0,1,ВАО,0,6438.509305
4,110,10.0,1700,17000,C,0,0,1,0,1,...,1710,335,104,706,0,0,1,ЦАО,0,4255.609941
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,30901,1563.0,8642,13507446,C,2,2,0,1,0,...,1963,186,29,597,0,0,1,ЦАО,1,1374.498493
391,31285,1708.0,8896,15194368,C,3,0,0,1,0,...,1963,186,29,597,0,0,1,ЦАО,1,1374.498493
392,31355,1708.0,9150,15628200,C,4,0,0,1,0,...,1963,186,29,597,0,0,1,ЦАО,1,1374.498493
393,31553,1644.0,10477,17224872,C,4,4,0,0,0,...,632,11,28,213,0,0,1,ЦАО,0,3460.954391


In [4]:
dataset.columns

Index(['Unnamed: 0', 'space_m2', 'price_per_m2', 'total_price', 'office_class',
       'floor', 'maxfloor', 'isCU', 'isNonCU', 'isEE', 'avg_latitude',
       'avg_longtitude', 'isMall', 'isOfficeCenter', 'inTTK', 'target',
       'minDistSubway', 'cafe_500', 'cafe_1000', 'cafe_2000', 'fast_food_500',
       'fast_food_1000', 'fast_food_2000', 'restaurant_500', 'restaurant_1000',
       'restaurant_2000', 'bank_500', 'bank_1000', 'bank_2000',
       'supermarket_500', 'supermarket_1000', 'supermarket_2000',
       'clothes_500', 'clothes_1000', 'clothes_2000', 'cosmetics_500',
       'cosmetics_1000', 'cosmetics_2000', 'electronics_500',
       'electronics_1000', 'electronics_2000', 'mall_500', 'mall_1000',
       'mall_2000', 'histplaces_500', 'histplaces_1000', 'histplaces_2000',
       'coords_str', 'TotalOfficeCnt_500', 'TotalOfficeCnt_1000',
       'TotalOfficeCnt_2000', 'classACnt_500', 'classACnt_1000',
       'classACnt_2000', 'isMoscowCity', 'isSkolkovo', 'inMKAD', 'AO',
     

In [5]:
### Размечаем уникальные здания
dataset['coords_str'] = [str(round(dataset.at[i, 'avg_latitude'], 6)) + '_' + str(round(dataset.at[i, 'avg_longtitude'], 6)) for i in dataset.index]
# число уникальных точек
print('Уникальных адресов:', len(set(dataset['coords_str'])))

l1 = sorted(list(set(dataset['coords_str'])))

address_to_num = {val: i for i, val in enumerate(l1)}
dataset['address_index'] = [address_to_num[i] for i in dataset['coords_str']]
print(min(dataset.address_index), max(dataset.address_index))
dataset

Уникальных адресов: 229
0 228


,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter,address_index
0,28,7.0,1357,9500,C,1,1,0,0,0,...,0,35,6,0,0,1,CВАО,0,7592.842639,208
1,31,12.3,976,12000,C,3,4,0,1,0,...,0,1,77,0,0,1,ЮВАО,0,5449.520429,49
2,99,12.0,1167,14000,C,3,3,0,0,0,...,0,8,46,0,0,1,СЗАО,0,7381.297810,125
3,109,15.0,1100,16500,C,2,4,1,0,1,...,18,3,313,0,0,1,ВАО,0,6438.509305,148
4,110,10.0,1700,17000,C,0,0,1,0,1,...,335,104,706,0,0,1,ЦАО,0,4255.609941,161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
390,30901,1563.0,8642,13507446,C,2,2,0,1,0,...,186,29,597,0,0,1,ЦАО,1,1374.498493,107
391,31285,1708.0,8896,15194368,C,3,0,0,1,0,...,186,29,597,0,0,1,ЦАО,1,1374.498493,107
392,31355,1708.0,9150,15628200,C,4,0,0,1,0,...,186,29,597,0,0,1,ЦАО,1,1374.498493,107
393,31553,1644.0,10477,17224872,C,4,4,0,0,0,...,11,28,213,0,0,1,ЦАО,0,3460.954391,117


### Строим катбуст по 10 разным разбиениям, считаем средние метрики

In [6]:
from catboost import CatBoostRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [7]:
# список фичей
features = [
    # В1: Помещение + здание, 
    'floor', 'maxfloor', 
    'isCU', 'isNonCU', 'isEE', 'isOfficeCenter', 'isMall',
    'inTTK', 'inGardenRing', 'AO',
    'space_m2',
    # В2: В1 + метро
    'minDistSubway',
    # В3: В2 + окружение
    'cafe_500', 'cafe_1000', 'cafe_2000',
    'fast_food_500', 'fast_food_1000', 'fast_food_2000', 
    'restaurant_500', 'restaurant_1000', 'restaurant_2000', 
    'bank_500', 'bank_1000', 'bank_2000', 
    'supermarket_500', 'supermarket_1000', 'supermarket_2000', 
    'clothes_500', 'clothes_1000', 'clothes_2000',
    'cosmetics_500', 'cosmetics_1000', 'cosmetics_2000',
    'electronics_500', 'electronics_1000', 'electronics_2000',
    'mall_500', 'mall_1000', 'mall_2000', 
    'histplaces_500', 'histplaces_1000', 'histplaces_2000',
    'distanceToCenter',
]
cat_features = ['isCU', 'isNonCU', 'isEE', 'inTTK', 'isOfficeCenter', 'isMall', 'AO']

In [8]:
mses = []
maes = []
absolutemaes = []
r2s = []

for seed in random_seeds:
    # Создаем генератор
    rng = np.random.default_rng(seed=seed)

    # берем 50 уникальных адресов
    test_index = rng.choice(max(dataset.address_index), size=50, replace=False)
    print(test_index[0:5])
    
    # делим датасет на тестовый и проверочный
    X_train = dataset.loc[~dataset['address_index'].isin(test_index)][features]
    y_train = dataset.loc[~dataset['address_index'].isin(test_index)]['target']
    X_test = dataset.loc[dataset['address_index'].isin(test_index)][features]
    y_test = dataset.loc[dataset['address_index'].isin(test_index)]['target']
    y_testRUR = dataset.loc[dataset['address_index'].isin(test_index)]['price_per_m2']
    print(y_train.shape, y_test.shape)
    
    # создаем модель
    model = CatBoostRegressor(
        iterations=400,
        learning_rate=0.015,
        random_state=seed,
        depth=10,
        loss_function='RMSE',
        cat_features = cat_features,
        verbose=0,
        l2_leaf_reg=2.0
    )
    
    # учим  модель
    model.fit(X_train, y_train)
    
    # получаем предикты
    predictions = model.predict(X_test)
    
    # MSE, MAE
    mses.append(mean_squared_error(y_test, predictions))
    maes.append(mean_absolute_error(y_test, predictions))
    
    # AbsoluteMAE
    absolutemaes.append(mean_absolute_error(np.exp(predictions), y_testRUR))
    
    # R2
    r2s.append(model.score(X_test, y_test))

[136  63   4  84 223]
(316,) (79,)
[ 82 100  89  15 177]
(303,) (92,)
[179  88  29 109 138]
(301,) (94,)
[166 132  72 126   0]
(313,) (82,)
[ 76  87  54 118  88]
(307,) (88,)
[ 37 184 146 156 155]
(324,) (71,)
[183  33  21 121  25]
(306,) (89,)
[ 61 171 160  59  23]
(308,) (87,)
[201 161  63 213  41]
(322,) (73,)
[ 95  56 217  43 196]
(322,) (73,)


In [9]:
print(f'#--- AVG MSE: {np.mean(mses):.5f}' )
print(f'#--- AVG MAE: {np.mean(maes):.5f}')
print(f'#ABS AVG MAE: {np.mean(absolutemaes):.5f}')
print(f'#--- AVG  R2: {np.mean(r2s):.5f}')

#--- AVG MSE: 0.16014
#--- AVG MAE: 0.28393
#ABS AVG MAE: 891.66106
#--- AVG  R2: 0.43698


In [10]:
### Хар-ки офиса + признаки локации
#--- AVG MSE: 0.18955
#--- AVG MAE: 0.32999
#ABS AVG MAE: 981.23984
#--- AVG  R2: 0.33240
### -//- + метро
#--- AVG MSE: 0.17522
#--- AVG MAE: 0.31309
#ABS AVG MAE: 937.93597
#--- AVG  R2: 0.37910
### -//- + окружение
#--- AVG MSE: 0.16014
#--- AVG MAE: 0.28393
#ABS AVG MAE: 891.66106
#--- AVG  R2: 0.43698

In [11]:
# Выводим значимость фичей
print("\nFeature Importances:")
dict(zip(features, model.get_feature_importance()))


Feature Importances:


{'floor': 1.5104008138130458,
 'maxfloor': 3.644347023684719,
 'isCU': 0.26230651261057075,
 'isNonCU': 0.3397160472849063,
 'isEE': 0.08479958811331033,
 'isOfficeCenter': 0.9752333064058049,
 'isMall': 0.001081381701689442,
 'inTTK': 0.08235303101846053,
 'inGardenRing': 0.6726935282151043,
 'AO': 5.673129076865406,
 'space_m2': 6.934574276922375,
 'minDistSubway': 7.082081918979804,
 'cafe_500': 5.053475348193198,
 'cafe_1000': 1.505939226166233,
 'cafe_2000': 1.8616783049532482,
 'fast_food_500': 2.6280545670663473,
 'fast_food_1000': 3.4191151777118804,
 'fast_food_2000': 1.9642452858863877,
 'restaurant_500': 3.108424977371909,
 'restaurant_1000': 3.4412735694580765,
 'restaurant_2000': 4.694206586993675,
 'bank_500': 2.318999907162808,
 'bank_1000': 2.060596671613756,
 'bank_2000': 3.3173972096776216,
 'supermarket_500': 3.183348122715092,
 'supermarket_1000': 4.526149917912041,
 'supermarket_2000': 3.437116969347873,
 'clothes_500': 2.1244466195892997,
 'clothes_1000': 2.974950

In [12]:
# Выводим топ-7 фичей по значимости для последнего разбиения
fi_df = pd.DataFrame({
    'Feature': X_test.columns,
    'Importance': model.get_feature_importance()
})
top_15_features = fi_df.sort_values(by='Importance', ascending=False).head(7)
display(top_15_features)

,Feature,Importance
11,minDistSubway,7.082082
10,space_m2,6.934574
9,AO,5.673129
12,cafe_500,5.053475
20,restaurant_2000,4.694207
25,supermarket_1000,4.526150
1,maxfloor,3.644347


### Строим линейную модель из sklearn

In [13]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

In [14]:
datasetLin = dataset
datasetLin.head()

,Unnamed: 0,space_m2,price_per_m2,total_price,office_class,floor,maxfloor,isCU,isNonCU,isEE,...,classACnt_500,classACnt_1000,classACnt_2000,isMoscowCity,isSkolkovo,inMKAD,AO,inGardenRing,distanceToCenter,address_index
0,28,7.0,1357,9500,C,1,1,0,0,0,...,0,35,6,0,0,1,CВАО,0,7592.842639,208
1,31,12.3,976,12000,C,3,4,0,1,0,...,0,1,77,0,0,1,ЮВАО,0,5449.520429,49
2,99,12.0,1167,14000,C,3,3,0,0,0,...,0,8,46,0,0,1,СЗАО,0,7381.297810,125
3,109,15.0,1100,16500,C,2,4,1,0,1,...,18,3,313,0,0,1,ВАО,0,6438.509305,148
4,110,10.0,1700,17000,C,0,0,1,0,1,...,335,104,706,0,0,1,ЦАО,0,4255.609941,161


In [15]:
datasetLin['nl_total_price'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['total_price'])]
datasetLin['nl_floor'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['floor'])]
datasetLin['nl_space_m2'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['space_m2'])]
datasetLin['nl_minDistSubway'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['minDistSubway'])]
datasetLin['nl_maxfloor'] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin['maxfloor'])]
env = [
    'cafe_500', 'cafe_1000', 'cafe_2000', 'fast_food_500', 'fast_food_1000', 'fast_food_2000', 'restaurant_500',
    'restaurant_1000', 'restaurant_2000', 'bank_500', 'bank_1000', 'bank_2000', 'supermarket_500', 'supermarket_1000', 'supermarket_2000', 'clothes_500', 'clothes_1000', 'clothes_2000',
    'cosmetics_500', 'cosmetics_1000', 'cosmetics_2000', 'electronics_500', 'electronics_1000', 'electronics_2000',
    'mall_500', 'mall_1000', 'mall_2000', 'histplaces_500', 'histplaces_1000', 'histplaces_2000', 'distanceToCenter'
]
for i in env:
    datasetLin['nl_' + i] = [np.log(max(0.0, i) + 1.0) for i in list(datasetLin[i])]
datasetLin = pd.get_dummies(datasetLin, columns=['office_class'], dtype=int)
datasetLin = pd.get_dummies(datasetLin, columns=['AO'], dtype=int)

linearFeatures = [
    'nl_floor', 'nl_maxfloor', 'isCU', 'isNonCU', 'isEE', 'isOfficeCenter', 'isMall',
    'inTTK', 'inGardenRing',
    'AO_CВАО', 'AO_ВАО', 'AO_ЗАО', 'AO_САО', 'AO_СЗАО', 'AO_ЮАО', 'AO_ЮВАО', 'AO_ЮЗАО',
    'nl_space_m2',
    # + метро
    'nl_minDistSubway',
    # + окружение
    'nl_cafe_500', 'nl_cafe_1000', 'nl_cafe_2000',
    'nl_fast_food_500', 'nl_fast_food_1000', 'nl_fast_food_2000', 
    'nl_restaurant_500', 'nl_restaurant_1000', 'nl_restaurant_2000', 
    'nl_bank_500', 'nl_bank_1000', 'nl_bank_2000', 
    'nl_supermarket_500', 'nl_supermarket_1000', 'nl_supermarket_2000',
    'nl_clothes_500', 'nl_clothes_1000', 'nl_clothes_2000',
    'nl_cosmetics_500', 'nl_cosmetics_1000', 'nl_cosmetics_2000',
    'nl_electronics_500', 'nl_electronics_1000', 'nl_electronics_2000',
    'nl_mall_500', 'nl_mall_1000', 'nl_mall_2000',
    'nl_histplaces_500', 'nl_histplaces_1000', 'nl_histplaces_2000',
    'nl_distanceToCenter'
]
exceptFeatures = []#['total_price', 'floor', 'minDistSubway', 'maxfloor'] + env
for i in features:
    if i not in linearFeatures + exceptFeatures:
        print(i)

floor
maxfloor
AO
space_m2
minDistSubway
cafe_500
cafe_1000
cafe_2000
fast_food_500
fast_food_1000
fast_food_2000
restaurant_500
restaurant_1000
restaurant_2000
bank_500
bank_1000
bank_2000
supermarket_500
supermarket_1000
supermarket_2000
clothes_500
clothes_1000
clothes_2000
cosmetics_500
cosmetics_1000
cosmetics_2000
electronics_500
electronics_1000
electronics_2000
mall_500
mall_1000
mall_2000
histplaces_500
histplaces_1000
histplaces_2000
distanceToCenter


In [16]:
mses = []
maes = []
absolutemaes = []
r2s = []

for seed in random_seeds:
    # Создаем генератор
    rng = np.random.default_rng(seed=seed)

    # берем 50 уникальных адресов
    test_index = rng.choice(max(datasetLin.address_index), size=50, replace=False)
    print(test_index[0:5])
    
    X_train = datasetLin.loc[~datasetLin['address_index'].isin(test_index)][linearFeatures]
    y_train = datasetLin.loc[~datasetLin['address_index'].isin(test_index)]['target']
    X_test = datasetLin.loc[datasetLin['address_index'].isin(test_index)][linearFeatures]
    y_test = datasetLin.loc[datasetLin['address_index'].isin(test_index)]['target']
    y_testRUR = dataset.loc[datasetLin['address_index'].isin(test_index)]['price_per_m2']
    
    # учим модель
    model = LinearRegression()
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    # считаем метрики
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    absolutemae = mean_absolute_error(np.exp(y_pred), y_testRUR)
    
    mses.append(mse)
    r2s.append(r2)
    maes.append(mae)
    absolutemaes.append(absolutemae)

[136  63   4  84 223]
[ 82 100  89  15 177]
[179  88  29 109 138]
[166 132  72 126   0]
[ 76  87  54 118  88]
[ 37 184 146 156 155]
[183  33  21 121  25]
[ 61 171 160  59  23]
[201 161  63 213  41]
[ 95  56 217  43 196]


In [17]:
print(f'#--- AVG MSE: {np.mean(mses):.5f}' )
print(f'#--- AVG MAE: {np.mean(maes):.5f}')
print(f'#ABS AVG MAE: {np.mean(absolutemaes):.5f}')
print(f'#--- AVG  R2: {np.mean(r2s):.5f}')

#--- AVG MSE: 0.22319
#--- AVG MAE: 0.35374
#ABS AVG MAE: 1034.18851
#--- AVG  R2: 0.20570


In [18]:
### Хар-ки офиса + признаки локации
#--- AVG MSE: 0.23358
#--- AVG MAE: 0.36139
#ABS AVG MAE: 1046.95284
#--- AVG  R2: 0.18038
### -//- + метро
#--- AVG MSE: 0.22490
#--- AVG MAE: 0.35015
#ABS AVG MAE: 1027.92531
#--- AVG  R2: 0.21411
### -//- + окружение
#--- AVG MSE: 0.22319
#--- AVG MAE: 0.35374
#ABS AVG MAE: 1034.18851
#--- AVG  R2: 0.20570

In [19]:
# коэфтцтенты
coefficients = model.coef_
intercept = model.intercept_

print(f"Intercept: {intercept}")
for i in range(len(coefficients)):
    if abs(coefficients[i]) > 0.12:
        print(round(coefficients[i], 5), linearFeatures[i])

Intercept: 8.075827146840247
-0.15747 isMall
-0.13597 inTTK
-0.43145 inGardenRing
-0.30625 AO_ВАО
0.207 AO_ЗАО
-0.2781 AO_ЮАО
0.17014 nl_restaurant_500
0.21538 nl_restaurant_2000
-0.83446 nl_mall_500
0.61803 nl_histplaces_500
0.14736 nl_histplaces_2000


In [20]:
coef_df = pd.DataFrame(
    {
        "Feature": X_train.columns,
        "Coefficient": model.coef_,  # Use model.coef_[0] for logistic regression
    }
)

# 2. Sort by absolute value to find the strongest positive and negative impacts
coef_df["Abs_Coefficient"] = coef_df["Coefficient"].abs()
top = coef_df.sort_values(by="Abs_Coefficient", ascending=False)

# 3. Display the result
display(top[["Feature", "Coefficient"]])

,Feature,Coefficient
43,nl_mall_500,-0.834464
46,nl_histplaces_500,0.618028
8,inGardenRing,-0.431446
10,AO_ВАО,-0.306249
14,AO_ЮАО,-0.278100
27,nl_restaurant_2000,0.215375
11,AO_ЗАО,0.206999
25,nl_restaurant_500,0.170139
6,isMall,-0.157465
48,nl_histplaces_2000,0.147357


In [21]:
tempdf = pd.DataFrame()
for i in ['nl_minDistSubway', 'nl_restaurant_500', 'nl_restaurant_1000', 'nl_restaurant_2000',
         'nl_supermarket_500', 'nl_supermarket_1000', 'nl_supermarket_2000',
         'nl_fast_food_500', 'nl_fast_food_1000', 'nl_fast_food_2000',
         'nl_cafe_500', 'nl_cafe_1000', 'nl_cafe_2000',
         'nl_histplaces_500', 'nl_histplaces_1000', 'nl_histplaces_2000']:
    t = top.loc[top['Feature'] == i].reset_index(drop=True)
    tempdf = pd.concat([tempdf, t])
tempdf[['Feature', 'Coefficient']]
print(tempdf[['Coefficient']].to_string(index=False))

 Coefficient
    0.040426
    0.170139
   -0.007586
    0.215375
    0.068492
   -0.043937
   -0.006285
   -0.105559
   -0.032829
   -0.112722
    0.086374
    0.010679
   -0.119538
    0.618028
   -0.110449
    0.147357
